# Forex Anomaly Detection — Exploratory Data Analysis

Run `python main.py generate` and `python main.py train` before executing this notebook.

**Sections**
1. Load data
2. Portal events overview
3. Trade events overview
4. Feature matrix analysis
5. Anomaly score distribution
6. Feature importance (excess-ratio ranking)
7. Normal vs Anomalous user comparison

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make src/ importable
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

from src.model import AnomalyDetector, MODEL_FEATURES

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Ready.')

## 1. Load data

In [ ]:
DATA = ROOT / 'data'

portal_df  = pd.read_parquet(DATA / 'portal_events.parquet')
trades_df  = pd.read_parquet(DATA / 'trade_events.parquet')
features_df = pd.read_parquet(DATA / 'features.parquet')

print(f'Portal events : {portal_df.shape}')
print(f'Trade events  : {trades_df.shape}')
print(f'Feature matrix: {features_df.shape}')

## 2. Portal events overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Event type distribution
portal_df['event_type'].value_counts().plot.bar(ax=axes[0], color='steelblue')
axes[0].set_title('Event Type Distribution')
axes[0].set_xlabel('')

# Events per user
evts_per_user = portal_df.groupby('user_id').size()
axes[1].hist(evts_per_user, bins=40, color='teal', edgecolor='none')
axes[1].set_title('Portal Events per User')
axes[1].set_xlabel('Event count')

# Region distribution
portal_df['region'].value_counts().plot.bar(ax=axes[2], color='coral')
axes[2].set_title('Login Regions')
axes[2].set_xlabel('')

plt.tight_layout()
plt.show()

## 3. Trade events overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

# Trade size distribution (log scale)
axes[0].hist(np.log1p(trades_df['trade_size']), bins=50, color='steelblue', edgecolor='none')
axes[0].set_title('log(trade_size + 1) Distribution')

# Leverage distribution
trades_df['leverage'].value_counts().sort_index().plot.bar(ax=axes[1], color='teal')
axes[1].set_title('Leverage Distribution')

# Margin usage
axes[2].hist(trades_df['margin_usage_pct'], bins=50, color='coral', edgecolor='none')
axes[2].axvline(80, color='red', linestyle='--', label='80% threshold')
axes[2].legend()
axes[2].set_title('Margin Usage (%)')

# Currency pair popularity
trades_df['currency_pair'].value_counts().plot.bar(ax=axes[3], color='purple')
axes[3].set_title('Currency Pair Popularity')

# Trade duration
axes[4].hist(np.log1p(trades_df['trade_duration_s']), bins=50, color='olive', edgecolor='none')
axes[4].set_title('log(trade_duration_s + 1)')

# Normal vs Anomalous trade size
normal = trades_df[~trades_df['is_anomalous']]['trade_size']
anomalous = trades_df[trades_df['is_anomalous']]['trade_size']
axes[5].hist(np.log1p(normal), bins=40, alpha=0.6, label='Normal', color='steelblue')
axes[5].hist(np.log1p(anomalous), bins=40, alpha=0.6, label='Anomalous', color='red')
axes[5].set_title('Trade Size: Normal vs Anomalous')
axes[5].legend()

plt.tight_layout()
plt.show()

## 4. Feature matrix analysis

In [ ]:
print('Feature matrix summary:')
display(features_df[MODEL_FEATURES].describe().round(2))

In [ ]:
# Correlation heatmap of model features
corr = features_df[MODEL_FEATURES].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    corr, mask=mask, annot=False, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.3,
    ax=ax
)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Anomaly score distribution

In [ ]:
detector = AnomalyDetector(str(ROOT / 'configs/config.yaml')).load()
scores_df = detector.score(features_df)

print(f"Anomalies detected : {scores_df['is_anomaly'].sum()} / {len(scores_df)}")
print(f"Score mean         : {scores_df['anomaly_score'].mean():.4f}")
print(f"Score std          : {scores_df['anomaly_score'].std():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Score histogram
normal_scores = scores_df[~scores_df['is_anomaly']]['anomaly_score']
anomaly_scores = scores_df[scores_df['is_anomaly']]['anomaly_score']

axes[0].hist(normal_scores, bins=40, alpha=0.7, label='Normal', color='steelblue')
axes[0].hist(anomaly_scores, bins=40, alpha=0.7, label='Anomalous', color='red')
axes[0].axvline(0.65, color='orange', linestyle='--', label='Alert threshold 0.65')
axes[0].set_title('Anomaly Score Distribution')
axes[0].set_xlabel('Anomaly Score')
axes[0].legend()

# Top 20 users by score
top20 = scores_df.nlargest(20, 'anomaly_score')
colors = ['red' if a else 'steelblue' for a in top20['is_anomaly']]
axes[1].barh(top20['user_id'], top20['anomaly_score'], color=colors)
axes[1].set_title('Top 20 Users by Anomaly Score')
axes[1].set_xlabel('Anomaly Score')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Feature importance (excess-ratio ranking)

In [ ]:
from src.explainability import RuleBasedExplainer, RULE_DEFINITIONS

explainer = RuleBasedExplainer(top_k=3).fit(features_df)
explained = explainer.explain_batch(features_df, scores_df)

# Count how often each feature fires
from collections import Counter
feat_counts = Counter()
for feats in explained['features_triggered']:
    feat_counts.update(feats)

feat_series = pd.Series(feat_counts).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
feat_series.plot.bar(ax=ax, color='coral', edgecolor='none')
ax.set_title('Most Frequently Triggered Features in Anomalies')
ax.set_ylabel('Alert count')
ax.set_xlabel('')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 7. Normal vs Anomalous user comparison

In [ ]:
merged = features_df.merge(scores_df[['user_id', 'is_anomaly', 'anomaly_score']], on='user_id')

key_features = [
    'max_trades_5min', 'max_margin_usage', 'trade_frequency_per_hour',
    'unique_regions', 'withdrawal_deposit_ratio', 'max_trade_size'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    normal_vals = merged[~merged['is_anomaly']][feat]
    anom_vals   = merged[merged['is_anomaly']][feat]
    
    axes[i].hist(np.log1p(normal_vals), bins=30, alpha=0.6, label='Normal', color='steelblue')
    axes[i].hist(np.log1p(anom_vals),   bins=30, alpha=0.6, label='Anomalous', color='red')
    axes[i].set_title(f'log({feat}+1)')
    axes[i].legend(fontsize=8)

plt.suptitle('Normal vs Anomalous Users — Key Feature Distributions', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: median values by group
summary = merged.groupby('is_anomaly')[key_features].median().round(2)
summary.index = ['Normal', 'Anomalous']
display(summary.T)